In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import pickle
import umap
import hdbscan



c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [237]:
df=pd.read_excel('../input_data/Feb28TGASP_Base.xlsx')

In [254]:
dfn=pd.read_excel('../input_data/fina_feb_25.xlsx')

In [255]:
dfn.shape

(302, 20)

In [ ]:
df["Parent Industry"].unique()

array(['Retail & Consumer', 'Information & Technology', 'Healthcare',
       'Manufacturing', 'Engineering', 'Professional Services',
       'Education', 'Finance'], dtype=object)

In [20]:
std2d1=np.load('../vector/feb25base.npy')

In [156]:
rows_of_users= []
for i,j in df.iterrows():
    sentence  = f"An {j['Job_Seniority']} in {j['Job_Function']} from the {j['Parent Industry']} industry. Their key areas of interest include {j['aoi_1']}, {j['aoi_2']}, {j['aoi_3']}."
    rows_of_users.append(sentence)

In [256]:
rows_of_usersn= []
for i,j in dfn.iterrows():
    sentence  = f"An {j['Job_Seniority']} in {j['Job_Function']} from the {j['Parent Industry']} industry. Their key areas of interest include {j['aoi_1']}, {j['aoi_2']}, {j['aoi_3']}."
    rows_of_usersn.append(sentence)

In [257]:
print(len(rows_of_usersn))

302


In [138]:
base = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [157]:
embedding = base.encode(rows_of_users, normalize_embeddings=True)

In [258]:
embeddingn = base.encode(rows_of_usersn, normalize_embeddings=True)


In [260]:
np.save('../vector/feb25.npy', embeddingn)

In [261]:
embeddingn

array([[ 0.01186802, -0.02480539,  0.04973789, ...,  0.00744042,
         0.02665696,  0.03238562],
       [ 0.06125262, -0.08243895,  0.03571336, ...,  0.00619081,
        -0.04291524,  0.02489005],
       [ 0.01290914,  0.02596411, -0.01946688, ..., -0.05208604,
        -0.04011492,  0.04401339],
       ...,
       [ 0.0208208 , -0.01351083,  0.01914846, ..., -0.03713532,
        -0.00191599,  0.0132809 ],
       [ 0.04192988, -0.0391866 , -0.0298844 , ..., -0.03421316,
         0.04397052,  0.00676906],
       [ 0.04933105, -0.07512295,  0.07295789, ..., -0.01298859,
        -0.02948759,  0.02728129]], shape=(302, 384), dtype=float32)

In [163]:
sr= StandardScaler()     


In [236]:
std= sr.fit_transform(embedding)  
stdn= sr.fit_transform(embeddingn)            
std2d= umap.UMAP(n_components=2,random_state=41).fit_transform(std)
std2dn= umap.UMAP(n_components=2,random_state=41).fit_transform(stdn)


c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [239]:
clusterer = hdbscan.HDBSCAN(min_samples=31,cluster_selection_epsilon=1.5,metric="euclidean",prediction_data=True)
labels = clusterer.fit_predict(std2d)
print(np.unique(labels).max()+1)
df["cluster"] = labels

17


c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [203]:
df.to_excel('../output/Feb28TGASP_Base_Clustered.xlsx', index=False)

In [ ]:


# Load your existing base file
df_base = pd.read_excel('../input_data/fina_feb_25.xlsx')

# Your new prediction results
predicted_labels, strengths = hdbscan.approximate_predict(clusterer, std2dn)

# Create new prediction columns to add to base file
new_predictions = pd.DataFrame({
    'new_predicted_cluster': predicted_labels,
    'new_prediction_strength': strengths,
    'new_cluster_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in predicted_labels],
    'new_confidence_level': [
        'High' if x > 0.7 else 'Medium' if x > 0.4 else 'Low' 
        for x in strengths
    ]
})

# Add new columns to existing base data
df_base_updated = pd.concat([df_base, new_predictions], axis=1)

# Save back to the same base file (or create new version)
df_base_updated.to_excel('../output/fen25base.xlsx', index=False)

print("New prediction columns added to base Excel file!")


New prediction columns added to base Excel file!


In [172]:
df_base_updated.to_excel('../output/fen25basepn.xlsx', index=False)

In [244]:
top_industries = (
    df_base_updated.groupby(['new_predicted_cluster','Parent Industry','Job_Seniority','Job_Function'])
      .size()
      .reset_index(name='count')
      .sort_values(['new_predicted_cluster', 'count'], ascending=[True, False])
      .groupby('new_predicted_cluster')
      .head(2)
)
# top_industries.to_excel("../output/cluster_11.xlsx")

In [245]:
top_industries

,new_predicted_cluster,Parent Industry,Job_Seniority,Job_Function,count
98,-1,Information & Technology,Executive,Business,73
117,-1,Information & Technology,Mid-Level,It/Technology,39
224,1,Information & Technology,Mid-Level,General Management,105
228,1,Professional Services,Mid-Level,General Management,14
233,2,Manufacturing,Mid-Level,General Management,29
232,2,Finance,Mid-Level,General Management,19
235,4,Information & Technology,Entry-Level,Business Analyst,13
236,4,Information & Technology,Senior-Level,Business Analyst,4
267,5,Information & Technology,CXO,Business,89
240,5,Consulting & Services,CXO,Business,38


In [247]:
# Step 1: Melt the AOI columns into a single column
df_melted =df_base_updated.melt(
    id_vars=['new_predicted_cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'],
    value_vars=['aoi_1', 'aoi_2', 'aoi_3'],
    var_name='aoi_type',
    value_name='aoi_value'
)
 
# Step 2: Group by relevant fields and count each AOI's frequency
aoi_counts = (
    df_melted.groupby(['new_predicted_cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function', 'aoi_value'])
    .size()
    .reset_index(name='count')
)
 
# Step 3: Sort and select top 2 AOIs per cluster
top_2_of_cluster = (
    aoi_counts.sort_values(['new_predicted_cluster', 'count'], ascending=[True, False])
              .groupby('new_predicted_cluster')
              .head(3)
)

In [248]:
top_2_of_cluster

,new_predicted_cluster,Parent Industry,Job_Seniority,Job_Function,aoi_value,count
204,-1,Information & Technology,Mid-Level,General Management,Artificial Intelligence,71
197,-1,Information & Technology,Mid-Level,General Management,Digital Transformation,67
192,-1,Information & Technology,Mid-Level,General Management,Customer Engagement,60
328,0,Finance,Mid-Level,General Management,Artificial Intelligence,73
314,0,Finance,Mid-Level,General Management,Digital Transformation,42
...,...,...,...,...,...,...
2272,20,Information & Technology,Mid-Level,General Management,Automation,31
2280,20,Information & Technology,Mid-Level,General Management,Digital Transformation,26
2333,21,Engineering,Mid-Level,General Management,Artificial Intelligence,12
2345,21,Information & Technology,Mid-Level,General Management,Data-Driven Insights,11


In [82]:
df.Job_Function.unique()


array(['It/Technology', 'General Management', 'Customer Service',
       'Business', 'Educator', 'Business Analyst', 'Product',
       'Quality Assurance', 'Finance', 'Operations', 'Design',
       'Consultant', 'Ai', 'Data', 'Engineering', 'Sales & Marketing',
       'Human Resources', 'Procurement', 'Law', 'Advisor', 'Data Science'],
      dtype=object)

In [199]:
df_base_updated["aoi_3"].unique()

array([' Leadership', ' Artificial Intelligence', ' Startups',
       ' Business', ' Strategy', ' Digital Transformation', ' Insights',
       ' Learning', ' Automation', ' Customer-Experience', ' Enterprise',
       ' Sales', ' Mentorship', ' Consulting', ' Fintech', ' Advisory',
       ' Transformation', ' Digital', ' Innovation',
       ' Enterprise Software', ' Education', ' Software',
       ' Cloud-Infrastructure', ' Marketing', ' Professional', 'Software',
       ' Supply Chain', ' Business Strategy', ' Digital-Transformation',
       ' BFSI', 'Strategy', ' Data'], dtype=object)

In [200]:
df["aoi_3"].unique()

array([' Customer Service', ' Leadership', ' Customer Engagement',
       ' Automation', ' Startups', ' Coaching', ' Data-Driven Insights',
       ' Digital Transformation', ' Digital Economy', ' Sales',
       ' Artificial Intelligence', ' Generative AI',
       ' Trust & Ethics in Tech', ' Customer-Centric',
       ' Innovation Strategy', ' Productivity Enhancement', ' Mentorship',
       ' Marketing', ' Decision Making', ' Personalized Experiences',
       ' Service Optimization', ' Analytics & Insights', ' Strategy',
       ' Hospitality', ' Technology', ' Cloud-Native Infrastructure',
       ' Enterprise Skills'], dtype=object)

In [201]:
# s3=np.load('../output/embeddingnewew.npy')
# s1=StandardScaler().fit_transform(s3)
# s2= umap.UMAP(n_components=2,random_state=41).fit_transform(s1)
# print(s2.shape, s3.shape,s1.shape)
# Check if new data is similar to training data
print("=== DATA DISTRIBUTION COMPARISON ===")
print(f"Training data range: {std2d.min():.3f} to {std2d.max():.3f}")
print(f"New data range: {std2dn.min():.3f} to {std2dn.max():.3f}")

# Check feature-wise statistics
for i in range(std2d.shape[1]):
    train_mean = std2d[:, i].mean()
    train_std = std2d[:, i].std()
    new_mean = std2dn[:, i].mean()
    new_std = std2dn[:, i].std()
    
    print(f"Feature {i}: Train μ={train_mean:.3f} σ={train_std:.3f}, New μ={new_mean:.3f} σ={new_std:.3f}")


=== DATA DISTRIBUTION COMPARISON ===
Training data range: -8.923 to 27.876
New data range: -7.950 to 18.978
Feature 0: Train μ=10.058 σ=7.276, New μ=10.118 σ=6.552
Feature 1: Train μ=6.810 σ=9.395, New μ=4.199 σ=6.532


In [115]:
# Generate soft clustering memberships instead of hard predictions
membership_vectors = hdbscan.membership_vector(clusterer, std2d1)

# Get most likely cluster for each point
soft_predictions = np.argmax(membership_vectors, axis=1)
max_memberships = np.max(membership_vectors, axis=1)

# Compare with approximate_predict
print(len(soft_predictions))
print("=== PREDICTION COMPARISON ===")
for i in range(min(10, len(predicted_labels))):
    print(f"Point {i}:")
    print(f"  approximate_predict: Cluster {predicted_labels[i]} (strength: {strengths[i]:.3f})")
    print(f"  membership_vector: Cluster {soft_predictions[i]} (membership: {max_memberships[i]:.3f})")
    print(len(soft_predictions))
dfa=df_base_updated
dfa['soft_predictions'] = soft_predictions
dfa.to_excel('../output/fen25basep_soft.xlsx', index=False)

302
=== PREDICTION COMPARISON ===
Point 0:
  approximate_predict: Cluster 11 (strength: 0.395)
  membership_vector: Cluster 11 (membership: 0.074)
302
Point 1:
  approximate_predict: Cluster -1 (strength: 0.000)
  membership_vector: Cluster 7 (membership: 0.005)
302
Point 2:
  approximate_predict: Cluster 1 (strength: 0.067)
  membership_vector: Cluster 1 (membership: 0.005)
302
Point 3:
  approximate_predict: Cluster 8 (strength: 0.088)
  membership_vector: Cluster 8 (membership: 0.009)
302
Point 4:
  approximate_predict: Cluster 1 (strength: 0.067)
  membership_vector: Cluster 1 (membership: 0.005)
302
Point 5:
  approximate_predict: Cluster -1 (strength: 0.000)
  membership_vector: Cluster 4 (membership: 0.004)
302
Point 6:
  approximate_predict: Cluster 11 (strength: 0.467)
  membership_vector: Cluster 11 (membership: 0.132)
302
Point 7:
  approximate_predict: Cluster 2 (strength: 0.159)
  membership_vector: Cluster 2 (membership: 0.027)
302
Point 8:
  approximate_predict: Cluster 

In [128]:
# Generate membership vectors (only for valid clusters, no noise)
membership_vectors = hdbscan.membership_vector(clusterer, std2d1)
valid_clusters = np.unique(labels[labels >= 0])  # Exclude -1 from mapping

print(f"Valid clusters for membership: {valid_clusters}")
print(f"Membership matrix shape: {membership_vectors.shape}")

# Get best cluster indices and membership strengths
cluster_indices = np.argmax(membership_vectors, axis=1)
max_memberships = np.max(membership_vectors, axis=1)

# Safely map indices to cluster labels
if membership_vectors.shape[1] > len(valid_clusters):
    cluster_indices = np.clip(cluster_indices, 0, len(valid_clusters)-1)

mapped_predictions = valid_clusters[cluster_indices]

# INCLUDE -1: Apply noise threshold to assign noise points
noise_threshold = 0.1  # Lower threshold = more noise points
final_membership_predictions = np.where(
    max_memberships < noise_threshold,
    -1,  # Assign as NOISE
    mapped_predictions  # Use cluster assignment
)

print("=== MEMBERSHIP PREDICTIONS WITH NOISE (-1) ===")
print(f"Unique clusters in membership: {sorted(np.unique(final_membership_predictions))}")
print(f"Noise points (-1): {np.sum(final_membership_predictions == -1)}")
print(f"Clustered points: {np.sum(final_membership_predictions != -1)}")

# Compare both methods including noise
print("\n=== COMPARISON WITH -1 INCLUDED ===")
for i in range(min(10, len(predicted_labels))):
    print(f"Point {i}:")
    print(f"  approximate_predict: Cluster {predicted_labels[i]} (strength: {strengths[i]:.3f})")
    print(f"  membership_vector: Cluster {final_membership_predictions[i]} (membership: {max_memberships[i]:.3f})")


Valid clusters for membership: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]
Membership matrix shape: (302, 22)
=== MEMBERSHIP PREDICTIONS WITH NOISE (-1) ===
Unique clusters in membership: [np.int64(-1), np.int64(1), np.int64(9), np.int64(11), np.int64(14), np.int64(16)]
Noise points (-1): 255
Clustered points: 47

=== COMPARISON WITH -1 INCLUDED ===
Point 0:
  approximate_predict: Cluster 11 (strength: 0.395)
  membership_vector: Cluster -1 (membership: 0.074)
Point 1:
  approximate_predict: Cluster -1 (strength: 0.000)
  membership_vector: Cluster -1 (membership: 0.005)
Point 2:
  approximate_predict: Cluster 1 (strength: 0.067)
  membership_vector: Cluster -1 (membership: 0.005)
Point 3:
  approximate_predict: Cluster 8 (strength: 0.088)
  membership_vector: Cluster -1 (membership: 0.009)
Point 4:
  approximate_predict: Cluster 1 (strength: 0.067)
  membership_vector: Cluster -1 (membership: 0.005)
Point 5:
  approximate_predict: Cluster -1 (strength: 0.000)
  membership_vec

In [129]:
# Create comprehensive DataFrame with -1 noise assignments
results_with_noise = pd.DataFrame({
    'sample_id': range(len(final_membership_predictions)),
    'approximate_cluster': predicted_labels,
    'approximate_strength': strengths,
    'membership_cluster': final_membership_predictions,  # Now includes -1
    'membership_strength': max_memberships,
    'both_methods_noise': (predicted_labels == -1) & (final_membership_predictions == -1),
    'methods_agree': predicted_labels == final_membership_predictions,
    'approximate_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in predicted_labels],
    'membership_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in final_membership_predictions]
})

# Add to your base Excel file with -1 included
df_base = pd.read_excel('../output/fen25base.xlsx')

df_base['membership_cluster_with_noise'] = final_membership_predictions
df_base['membership_strength'] = max_memberships
df_base['membership_is_noise'] = final_membership_predictions == -1
df_base['approximate_is_noise'] = predicted_labels == -1
df_base['both_methods_noise'] = (predicted_labels == -1) & (final_membership_predictions == -1)

# Save with noise included
df_base.to_excel('../output/fen25base_with_noise.xlsx', index=False)

print("✅ Results saved with -1 (noise) included in membership predictions!")


✅ Results saved with -1 (noise) included in membership predictions!


In [210]:

from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from scipy.spatial.distance import cdist
embedding_array=std2dn

# ========================================
# ENHANCED AOI ANALYSIS FOR EACH METHOD
# ========================================
def get_metho_specific_aoi_analysis(df, method_predictions, method_name):
    """
    Create AOI analysis for specific prediction method
    """
    # Create a copy of df with updated cluster predictions
    df_method = df.copy()
    df_method['cluster'] = method_predictions[:len(df)]  # Ensure same length
    
    # Step 1: Melt the AOI columns for this method
    df_melted = df_method.melt(
        id_vars=['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'],
        value_vars=['aoi_1', 'aoi_2', 'aoi_3'],
        var_name='aoi_type',
        value_name='aoi_value'
    )
    
    # Add method identifier
    df_melted['prediction_method'] = method_name
    
    # Step 2: Group by relevant fields and count each AOI's frequency
    aoi_counts = (
        df_melted.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function', 'aoi_value'])
        .size()
        .reset_index(name='count')
    )
    aoi_counts['prediction_method'] = method_name
    
    # Step 3: Sort and select top 3 AOIs per cluster for this method
    top_3_aoi = (
        aoi_counts.sort_values(['cluster', 'count'], ascending=[True, False])
                  .groupby('cluster')
                  .head(3)
    )
    
    return df_melted, aoi_counts, top_3_aoi

# ========================================
# ALL PREDICTION METHODS (Same as before)
# ========================================\
def get_method_specific_aoi_analysis(df, method_predictions, method_name):
    """
    Create AOI analysis for specific prediction method - handles length mismatch
    """
    print(f"Processing {method_name}: DF length={len(df)}, Predictions length={len(method_predictions)}")
    
    # Handle length mismatch between dataframe and predictions
    if len(method_predictions) < len(df):
        # Option 1: Use only the rows that have predictions
        df_method = df.iloc[:len(method_predictions)].copy()
        df_method['cluster'] = method_predictions
        
    elif len(method_predictions) > len(df):
        # Option 2: Trim predictions to match dataframe
        df_method = df.copy()
        df_method['cluster'] = method_predictions[:len(df)]
        
    else:
        # Perfect match
        df_method = df.copy()
        df_method['cluster'] = method_predictions
    
    print(f"  → Using {len(df_method)} rows for AOI analysis")
    
    # Step 1: Melt the AOI columns for this method
    df_melted = df_method.melt(
        id_vars=['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'],
        value_vars=['aoi_1', 'aoi_2', 'aoi_3'],
        var_name='aoi_type',
        value_name='aoi_value'
    )
    
    # Add method identifier
    df_melted['prediction_method'] = method_name
    
    # Step 2: Group by relevant fields and count each AOI's frequency
    aoi_counts = (
        df_melted.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function', 'aoi_value'])
        .size()
        .reset_index(name='count')
    )
    aoi_counts['prediction_method'] = method_name
    
    # Step 3: Sort and select top 3 AOIs per cluster for this method
    top_3_aoi = (
        aoi_counts.sort_values(['cluster', 'count'], ascending=[True, False])
                  .groupby('cluster')
                  .head(3)
    )
    
    return df_melted, aoi_counts, top_3_aoi

def get_approximate_predictions(clusterer, embedding_array):
    predicted_labels, strengths = hdbscan.approximate_predict(clusterer, embedding_array)
    
    return pd.DataFrame({
        'point_id': range(len(predicted_labels)),
        'predicted_cluster': predicted_labels,
        'prediction_strength': strengths,
        'cluster_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in predicted_labels],
        'confidence_level': ['High' if x > 0.7 else 'Medium' if x > 0.4 else 'Low' for x in strengths],
        'method': 'approximate_predict'
    }), predicted_labels

def get_membership_predictions(clusterer, embedding_array, labels):
    membership_vectors = hdbscan.membership_vector(clusterer, embedding_array)
    valid_clusters = np.unique(labels[labels >= 0])
    
    cluster_indices = np.argmax(membership_vectors, axis=1)
    max_memberships = np.max(membership_vectors, axis=1)
    
    if membership_vectors.shape[1] > len(valid_clusters):
        cluster_indices = np.clip(cluster_indices, 0, len(valid_clusters)-1)
    
    mapped_predictions = valid_clusters[cluster_indices]
    noise_threshold = 0.15
    final_predictions = np.where(max_memberships < noise_threshold, -1, mapped_predictions)
    
    return pd.DataFrame({
        'point_id': range(len(final_predictions)),
        'predicted_cluster': final_predictions,
        'prediction_strength': max_memberships,
        'cluster_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in final_predictions],
        'confidence_level': ['High' if x > 0.7 else 'Medium' if x > 0.4 else 'Low' for x in max_memberships],
        'method': 'membership_vector'
    }), final_predictions

def get_knn_predictions(std2d, labels, embedding_array):
    knn_predictor = KNeighborsClassifier(n_neighbors=5, weights='distance')
    knn_predictor.fit(std2d, labels)
    
    knn_predictions = knn_predictor.predict(embedding_array)
    knn_probabilities = knn_predictor.predict_proba(embedding_array)
    knn_confidence = np.max(knn_probabilities, axis=1)
    
    return pd.DataFrame({
        'point_id': range(len(knn_predictions)),
        'predicted_cluster': knn_predictions,
        'prediction_strength': knn_confidence,
        'cluster_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in knn_predictions],
        'confidence_level': ['High' if x > 0.7 else 'Medium' if x > 0.4 else 'Low' for x in knn_confidence],
        'method': 'knn_based'
    }), knn_predictions

def get_core_predictions(std2d, labels, embedding_array):
    def get_cluster_cores(data, cluster_labels, min_samples=11):
        cluster_cores = {}
        for cluster_id in np.unique(cluster_labels):
            if cluster_id == -1:
                continue
            cluster_mask = cluster_labels == cluster_id
            cluster_points = data[cluster_mask]
            
            if len(cluster_points) < min_samples:
                cluster_cores[cluster_id] = cluster_points
                continue
                
            nbrs = NearestNeighbors(n_neighbors=min_samples).fit(cluster_points)
            distances, indices = nbrs.kneighbors(cluster_points)
            core_threshold = np.percentile(distances[:, -1], 25)
            core_mask = distances[:, -1] <= core_threshold
            cluster_cores[cluster_id] = cluster_points[core_mask]
        return cluster_cores
    
    cluster_cores = get_cluster_cores(std2d, labels)
    
    def predict_via_cores(new_points, cluster_cores, max_distance=2.0):
        predictions = []
        distances = []
        
        for point in new_points:
            min_dist = float('inf')
            best_cluster = -1
            
            for cluster_id, cores in cluster_cores.items():
                if len(cores) == 0:
                    continue
                core_distances = np.linalg.norm(cores - point, axis=1)
                min_core_dist = np.min(core_distances)
                
                if min_core_dist < min_dist:
                    min_dist = min_core_dist
                    best_cluster = cluster_id
            
            if min_dist > max_distance:
                best_cluster = -1
            
            predictions.append(best_cluster)
            distances.append(min_dist)
        
        return np.array(predictions), np.array(distances)
    
    core_predictions, core_distances = predict_via_cores(embedding_array, cluster_cores)
    core_confidence = 1.0 / (1.0 + core_distances)
    
    return pd.DataFrame({
        'point_id': range(len(core_predictions)),
        'predicted_cluster': core_predictions,
        'prediction_strength': core_confidence,
        'cluster_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in core_predictions],
        'confidence_level': ['High' if x > 0.7 else 'Medium' if x > 0.4 else 'Low' for x in core_confidence],
        'method': 'core_based'
    }), core_predictions

def get_boundary_predictions(std2d, labels, embedding_array):
    def boundary_based_predict(new_points, training_data, training_labels, percentile=10):
        predictions = []
        confidences = []
        
        for point in new_points:
            cluster_scores = {}
            
            for cluster_id in np.unique(training_labels):
                if cluster_id == -1:
                    continue
                cluster_mask = training_labels == cluster_id
                cluster_points = training_data[cluster_mask]
                distances = cdist([point], cluster_points)[0]
                cluster_score = np.percentile(distances, percentile)
                cluster_scores[cluster_id] = cluster_score
            
            if cluster_scores:
                best_cluster = min(cluster_scores.keys(), key=lambda k: cluster_scores[k])
                min_score = cluster_scores[best_cluster]
                confidence = 1.0 / (1.0 + min_score)
                
                if confidence < 0.3:
                    best_cluster = -1
                
                predictions.append(best_cluster)
                confidences.append(confidence)
            else:
                predictions.append(-1)
                confidences.append(0.0)
        
        return np.array(predictions), np.array(confidences)
    
    boundary_pred, boundary_conf = boundary_based_predict(embedding_array, std2d, labels)
    
    return pd.DataFrame({
        'point_id': range(len(boundary_pred)),
        'predicted_cluster': boundary_pred,
        'prediction_strength': boundary_conf,
        'cluster_type': [f'NOISE' if x == -1 else f'Cluster_{x}' for x in boundary_pred],
        'confidence_level': ['High' if x > 0.7 else 'Medium' if x > 0.4 else 'Low' for x in boundary_conf],
        'method': 'boundary_based'
    }), boundary_pred

# ========================================
# GENERATE ALL PREDICTIONS AND AOI ANALYSES
# ========================================

# Generate all predictions
approx_df, approx_predictions = get_approximate_predictions(clusterer, embedding_array)
membership_df, membership_predictions = get_membership_predictions(clusterer, embedding_array, labels)
knn_df, knn_predictions = get_knn_predictions(std2d, labels, embedding_array)
core_df, core_predictions = get_core_predictions(std2d, labels, embedding_array)
boundary_df, boundary_predictions = get_boundary_predictions(std2d, labels, embedding_array)

# Generate method-specific AOI analyses
methods_data = {
    'Original': (labels, 'original'),
    'Approximate': (approx_predictions, 'approximate_predict'),
    'Membership': (membership_predictions, 'membership_vector'),
    'KNN': (knn_predictions, 'knn_based'),
    'Core': (core_predictions, 'core_based'),
    'Boundary': (boundary_predictions, 'boundary_based')
}

# Store AOI results for each method
aoi_results = {}
for method_display_name, (predictions, method_key) in methods_data.items():
    melted, counts, top3 = get_method_specific_aoi_analysis(df, predictions, method_key)
    aoi_results[method_display_name] = {
        'melted': melted,
        'counts': counts, 
        'top3': top3
    }

# Create comparison of all methods
comparison_df = pd.DataFrame({
    'point_id': range(len(embedding_array)),
    'approximate_cluster': approx_predictions,
    'approximate_strength': approx_df['prediction_strength'],
    'membership_cluster': membership_predictions,
    'membership_strength': membership_df['prediction_strength'],
    'knn_cluster': knn_predictions,
    'knn_strength': knn_df['prediction_strength'],
    'core_cluster': core_predictions,
    'core_strength': core_df['prediction_strength'],
    'boundary_cluster': boundary_predictions,
    'boundary_strength': boundary_df['prediction_strength']
})

# Add feature columns
for i in range(embedding_array.shape[1]):
    comparison_df[f'feature_{i}'] = embedding_array[:, i]

# ========================================
# CREATE COMPREHENSIVE EXCEL FILE
# ========================================
with pd.ExcelWriter('../output/complete_clustering_with_method_aoi.xlsx') as writer:
    
    # Original database
    df.to_excel(writer, sheet_name='Original_Database', index=False)
    
    # Individual prediction method sheets
    approx_df.to_excel(writer, sheet_name='Approximate_Predict', index=False)
    membership_df.to_excel(writer, sheet_name='Membership_Vector', index=False)
    knn_df.to_excel(writer, sheet_name='KNN_Based', index=False)
    core_df.to_excel(writer, sheet_name='Core_Based', index=False)
    boundary_df.to_excel(writer, sheet_name='Boundary_Based', index=False)
    
    # Comparison of all methods
    comparison_df.to_excel(writer, sheet_name='All_Methods_Comparison', index=False)
    
    # METHOD-SPECIFIC AOI ANALYSIS SHEETS
    for method_name, aoi_data in aoi_results.items():
        # AOI Melted Data for each method
        aoi_data['melted'].to_excel(writer, sheet_name=f'AOI_Melted_{method_name}', index=False)
        
        # AOI Counts for each method
        aoi_data['counts'].to_excel(writer, sheet_name=f'AOI_Counts_{method_name}', index=False)
        
        # Top 3 AOI per cluster for each method
        aoi_data['top3'].to_excel(writer, sheet_name=f'Top3_AOI_{method_name}', index=False)
    
    # Summary statistics
    summary_stats = pd.DataFrame({
        'Method': ['Original', 'Approximate', 'Membership', 'KNN', 'Core', 'Boundary'],
        'Total_Points': [len(df), len(approx_df), len(membership_df), len(knn_df), len(core_df), len(boundary_df)],
        'Unique_Clusters': [
            len(np.unique(labels)),
            len(np.unique(approx_predictions)),
            len(np.unique(membership_predictions)),
            len(np.unique(knn_predictions)),
            len(np.unique(core_predictions)),
            len(np.unique(boundary_predictions))
        ],
        'Noise_Points': [
            np.sum(labels == -1),
            np.sum(approx_predictions == -1),
            np.sum(membership_predictions == -1),
            np.sum(knn_predictions == -1),
            np.sum(core_predictions == -1),
            np.sum(boundary_predictions == -1)
        ]
    })
    
    summary_stats.to_excel(writer, sheet_name='Methods_Summary', index=False)

print("✅ Complete Excel file created: '../output/complete_clustering_with_method_aoi.xlsx'")
print("\nSheets created:")
print("  📊 Basic Sheets:")
print("     - Original_Database")
print("     - Approximate_Predict, Membership_Vector, KNN_Based, Core_Based, Boundary_Based")
print("     - All_Methods_Comparison")
print("     - Methods_Summary")

print("\n  🎯 Method-Specific AOI Analysis (18 sheets):")
for method_name in aoi_results.keys():
    print(f"     - AOI_Melted_{method_name}")
    print(f"     - AOI_Counts_{method_name}")
    print(f"     - Top3_AOI_{method_name}")

print(f"\n📈 Total Sheets: {6 + len(aoi_results) * 3 + 2} sheets")

# Show AOI comparison summary
print(f"\n=== AOI ANALYSIS SUMMARY BY METHOD ===")
for method_name, aoi_data in aoi_results.items():
    unique_clusters = len(aoi_data['top3']['cluster'].unique())
    total_aoi_entries = len(aoi_data['melted'])
    print(f"{method_name:12}: {unique_clusters:2d} clusters, {total_aoi_entries:4d} AOI entries")


Processing original: DF length=2382, Predictions length=2382
  → Using 2382 rows for AOI analysis
Processing approximate_predict: DF length=2382, Predictions length=1569
  → Using 1569 rows for AOI analysis
Processing membership_vector: DF length=2382, Predictions length=1569
  → Using 1569 rows for AOI analysis
Processing knn_based: DF length=2382, Predictions length=1569
  → Using 1569 rows for AOI analysis
Processing core_based: DF length=2382, Predictions length=1569
  → Using 1569 rows for AOI analysis
Processing boundary_based: DF length=2382, Predictions length=1569
  → Using 1569 rows for AOI analysis
✅ Complete Excel file created: '../output/complete_clustering_with_method_aoi.xlsx'

Sheets created:
  📊 Basic Sheets:
     - Original_Database
     - Approximate_Predict, Membership_Vector, KNN_Based, Core_Based, Boundary_Based
     - All_Methods_Comparison
     - Methods_Summary

  🎯 Method-Specific AOI Analysis (18 sheets):
     - AOI_Melted_Original
     - AOI_Counts_Original


In [214]:
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
import numpy as np
import hdbscan

embedding_array = std2dn

# ========================================
# ENHANCED ANALYSIS FUNCTIONS
# ========================================
def get_top3_aoi_analysis(df, method_predictions, method_name, method_params):
    """
    Get Top 3 AOI analysis with method parameters
    """
    # Handle length mismatch
    df_method = df.copy()
    if len(method_predictions) < len(df):
        df_method = df.iloc[:len(method_predictions)].copy()
    
    df_method['cluster'] = method_predictions[:len(df_method)]
    
    # Melt AOI data
    df_melted = df_method.melt(
        id_vars=['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'],
        value_vars=['aoi_1', 'aoi_2', 'aoi_3'],
        var_name='aoi_type',
        value_name='aoi_value'
    )
    
    # Count AOI frequencies
    aoi_counts = (
        df_melted.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function', 'aoi_value'])
        .size()
        .reset_index(name='count')
    )
    
    # Get Top 3 AOIs per cluster
    top_3_aoi = (
        aoi_counts.sort_values(['cluster', 'count'], ascending=[True, False])
                  .groupby('cluster')
                  .head(3)
                  .reset_index(drop=True)
    )
    
    # Add method information
    top_3_aoi['prediction_method'] = method_name
    top_3_aoi['method_parameters'] = method_params
    
    return top_3_aoi

def get_top2_industry_analysis(df, method_predictions, method_name, method_params):
    """
    NEW FUNCTION: Get Top 2 Industry combinations per cluster
    """
    # Handle length mismatch
    df_method = df.copy()
    if len(method_predictions) < len(df):
        df_method = df.iloc[:len(method_predictions)].copy()
    
    df_method['cluster'] = method_predictions[:len(df_method)]
    
    # Group by cluster and industry combinations
    top_2_industries = (
        df_method.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'])
        .size()
        .reset_index(name='count')
        .sort_values(['cluster', 'count'], ascending=[True, False])
        .groupby('cluster')
        .head(2)
        .reset_index(drop=True)
    )
    
    # Add method information
    top_2_industries['prediction_method'] = method_name
    top_2_industries['method_parameters'] = method_params
    
    return top_2_industries

# ========================================
# FIXED PREDICTION METHODS
# ========================================
def get_approximate_predictions(clusterer, embedding_array):
    predicted_labels, strengths = hdbscan.approximate_predict(clusterer, embedding_array)
    params = f"min_samples={clusterer.min_samples}, epsilon={clusterer.cluster_selection_epsilon}"
    return predicted_labels, params

def get_membership_predictions(clusterer, embedding_array, labels):
    """FIXED: Correct shape comparison"""
    membership_vectors = hdbscan.membership_vector(clusterer, embedding_array)
    valid_clusters = np.unique(labels[labels >= 0])
    
    print(f"Membership vectors shape: {membership_vectors.shape}")
    print(f"Valid clusters count: {len(valid_clusters)}")
    
    cluster_indices = np.argmax(membership_vectors, axis=1)
    max_memberships = np.max(membership_vectors, axis=1)
    
    # FIX: Compare number of columns (shape[1]) with number of valid clusters
    if membership_vectors.shape[1] > len(valid_clusters):
        print(f"Clipping indices: membership has {membership_vectors.shape[1]} columns, but only {len(valid_clusters)} valid clusters")
        cluster_indices = np.clip(cluster_indices, 0, len(valid_clusters)-1)
    
    mapped_predictions = valid_clusters[cluster_indices]
    final_predictions = np.where(max_memberships < 0.15, -1, mapped_predictions)
    
    params = f"min_samples={clusterer.min_samples}, noise_threshold=0.15"
    return final_predictions, params

def get_knn_predictions(std2d, labels, embedding_array):
    knn_predictor = KNeighborsClassifier(n_neighbors=5, weights='distance')
    knn_predictor.fit(std2d, labels)
    knn_predictions = knn_predictor.predict(embedding_array)
    
    params = "n_neighbors=5, weights=distance"
    return knn_predictions, params

# ========================================
# ROBUST COMPARISON ANALYSIS FUNCTIONS
# ========================================
def create_aoi_comparison_analysis(original_top3, derived_methods_top3):
    """Compare AOI patterns between original and derived methods - with error handling"""
    comparison_results = []
    
    try:
        # Get original cluster AOI patterns
        original_patterns = {}
        for _, row in original_top3.iterrows():
            cluster = row['cluster']
            if cluster not in original_patterns:
                original_patterns[cluster] = []
            original_patterns[cluster].append(row['aoi_value'])
        
        # Compare each derived method
        for method_name, derived_top3 in derived_methods_top3.items():
            if method_name == 'Original':
                continue
            
            try:
                derived_patterns = {}
                for _, row in derived_top3.iterrows():
                    cluster = row['cluster']
                    if cluster not in derived_patterns:
                        derived_patterns[cluster] = []
                    derived_patterns[cluster].append(row['aoi_value'])
                
                # Calculate similarity
                matching_patterns = 0
                similar_patterns = 0
                total_clusters = len(original_patterns) if original_patterns else 1
                
                for orig_cluster, orig_aois in original_patterns.items():
                    orig_aoi_set = set(orig_aois[:3])
                    best_match_score = 0
                    
                    for der_cluster, der_aois in derived_patterns.items():
                        der_aoi_set = set(der_aois[:3])
                        union_size = len(orig_aoi_set.union(der_aoi_set))
                        if union_size > 0:
                            match_score = len(orig_aoi_set.intersection(der_aoi_set)) / union_size
                            best_match_score = max(best_match_score, match_score)
                    
                    if best_match_score >= 0.8:
                        matching_patterns += 1
                    elif best_match_score >= 0.4:
                        similar_patterns += 1
                
                effectiveness = (matching_patterns + 0.5 * similar_patterns) / total_clusters * 100 if total_clusters > 0 else 0
                
                comparison_results.append({
                    'method': method_name,
                    'analysis_type': 'AOI_Patterns',
                    'total_original_clusters': total_clusters,
                    'matching_patterns': matching_patterns,
                    'similar_patterns': similar_patterns,
                    'effectiveness_score': effectiveness
                })
                
            except Exception as e:
                print(f"Error processing AOI comparison for {method_name}: {e}")
                comparison_results.append({
                    'method': method_name,
                    'analysis_type': 'AOI_Patterns',
                    'total_original_clusters': 0,
                    'matching_patterns': 0,
                    'similar_patterns': 0,
                    'effectiveness_score': 0
                })
    
    except Exception as e:
        print(f"Error in AOI comparison analysis: {e}")
    
    return comparison_results

def create_industry_comparison_analysis(original_top2, derived_methods_top2):
    """Compare industry patterns between original and derived methods - with error handling"""
    comparison_results = []
    
    try:
        # Get original cluster industry patterns
        original_patterns = {}
        for _, row in original_top2.iterrows():
            cluster = row['cluster']
            if cluster not in original_patterns:
                original_patterns[cluster] = []
            pattern = f"{row['Parent Industry']}|{row['Job_Seniority']}|{row['Job_Function']}"
            original_patterns[cluster].append(pattern)
        
        # Compare each derived method
        for method_name, derived_top2 in derived_methods_top2.items():
            if method_name == 'Original':
                continue
            
            try:
                derived_patterns = {}
                for _, row in derived_top2.iterrows():
                    cluster = row['cluster']
                    if cluster not in derived_patterns:
                        derived_patterns[cluster] = []
                    pattern = f"{row['Parent Industry']}|{row['Job_Seniority']}|{row['Job_Function']}"
                    derived_patterns[cluster].append(pattern)
                
                # Calculate similarity
                matching_patterns = 0
                similar_patterns = 0
                total_clusters = len(original_patterns) if original_patterns else 1
                
                for orig_cluster, orig_industries in original_patterns.items():
                    orig_industry_set = set(orig_industries[:2])
                    best_match_score = 0
                    
                    for der_cluster, der_industries in derived_patterns.items():
                        der_industry_set = set(der_industries[:2])
                        union_size = len(orig_industry_set.union(der_industry_set))
                        if union_size > 0:
                            match_score = len(orig_industry_set.intersection(der_industry_set)) / union_size
                            best_match_score = max(best_match_score, match_score)
                    
                    if best_match_score >= 0.8:
                        matching_patterns += 1
                    elif best_match_score >= 0.4:
                        similar_patterns += 1
                
                effectiveness = (matching_patterns + 0.5 * similar_patterns) / total_clusters * 100 if total_clusters > 0 else 0
                
                comparison_results.append({
                    'method': method_name,
                    'analysis_type': 'Industry_Patterns',
                    'total_original_clusters': total_clusters,
                    'matching_patterns': matching_patterns,
                    'similar_patterns': similar_patterns,
                    'effectiveness_score': effectiveness
                })
                
            except Exception as e:
                print(f"Error processing Industry comparison for {method_name}: {e}")
                comparison_results.append({
                    'method': method_name,
                    'analysis_type': 'Industry_Patterns',
                    'total_original_clusters': 0,
                    'matching_patterns': 0,
                    'similar_patterns': 0,
                    'effectiveness_score': 0
                })
    
    except Exception as e:
        print(f"Error in Industry comparison analysis: {e}")
    
    return comparison_results

# ========================================
# MAIN EXECUTION - ROBUST WITH ERROR HANDLING
# ========================================

# Generate predictions with parameters
print("Generating predictions...")
original_params = f"min_samples={clusterer.min_samples}, epsilon={clusterer.cluster_selection_epsilon}"

try:
    approx_predictions, approx_params = get_approximate_predictions(clusterer, embedding_array)
    print("✅ Approximate predictions generated")
except Exception as e:
    print(f"❌ Error in approximate predictions: {e}")
    approx_predictions, approx_params = labels, "error_approximate"

try:
    membership_predictions, membership_params = get_membership_predictions(clusterer, embedding_array, labels)
    print("✅ Membership predictions generated")
except Exception as e:
    print(f"❌ Error in membership predictions: {e}")
    membership_predictions, membership_params = labels, "error_membership"

try:
    knn_predictions, knn_params = get_knn_predictions(std2d, labels, embedding_array)
    print("✅ KNN predictions generated")
except Exception as e:
    print(f"❌ Error in KNN predictions: {e}")
    knn_predictions, knn_params = labels, "error_knn"

# Method configuration
methods_config = {
    'Original': (labels, original_params),
    'Approximate': (approx_predictions, approx_params),
    'Membership': (membership_predictions, membership_params),
    'KNN': (knn_predictions, knn_params)
}

# Generate BOTH analyses for each method
print("Generating Top 3 AOI and Top 2 Industry analyses...")
top3_aoi_results = {}
top2_industry_results = {}

for method_name, (predictions, params) in methods_config.items():
    try:
        print(f"  Processing {method_name}...")
        
        # Top 3 AOI analysis
        top3_aoi = get_top3_aoi_analysis(df, predictions, method_name, params)
        top3_aoi_results[method_name] = top3_aoi
        
        # Top 2 Industry analysis
        top2_industry = get_top2_industry_analysis(df, predictions, method_name, params)
        top2_industry_results[method_name] = top2_industry
        
        print(f"  ✅ {method_name} analysis completed")
        
    except Exception as e:
        print(f"  ❌ Error processing {method_name}: {e}")
        # Create empty DataFrames as fallback
        top3_aoi_results[method_name] = pd.DataFrame()
        top2_industry_results[method_name] = pd.DataFrame()

# Create comparison analyses
print("Creating comparison analyses...")
try:
    aoi_comparisons = create_aoi_comparison_analysis(top3_aoi_results['Original'], top3_aoi_results)
    print("✅ AOI comparisons completed")
except Exception as e:
    print(f"❌ Error in AOI comparisons: {e}")
    aoi_comparisons = []

try:
    industry_comparisons = create_industry_comparison_analysis(top2_industry_results['Original'], top2_industry_results)
    print("✅ Industry comparisons completed")
except Exception as e:
    print(f"❌ Error in Industry comparisons: {e}")
    industry_comparisons = []

# Combine comparison results
all_comparisons = pd.DataFrame(aoi_comparisons + industry_comparisons)

# ========================================
# CREATE COMPREHENSIVE EXCEL OUTPUT
# ========================================
print("Creating Excel file...")
try:
    with pd.ExcelWriter('../output/complete_top3_aoi_top2_industry_analysis_fixed.xlsx') as writer:
        
        # Top 3 AOI sheets for each method
        for method_name, top3_data in top3_aoi_results.items():
            if len(top3_data) > 0:  # Only create sheet if data exists
                sheet_name = f'Top3_AOI_{method_name}'
                top3_data.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Top 2 Industry sheets for each method  
        for method_name, top2_data in top2_industry_results.items():
            if len(top2_data) > 0:  # Only create sheet if data exists
                sheet_name = f'Top2_Industry_{method_name}'
                top2_data.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Combined comparison sheets
        if len(top3_aoi_results) > 0:
            combined_aoi = pd.concat([data for data in top3_aoi_results.values() if len(data) > 0], ignore_index=True)
            if len(combined_aoi) > 0:
                combined_aoi.to_excel(writer, sheet_name='All_Methods_Top3_AOI_Combined', index=False)
        
        if len(top2_industry_results) > 0:
            combined_industry = pd.concat([data for data in top2_industry_results.values() if len(data) > 0], ignore_index=True)
            if len(combined_industry) > 0:
                combined_industry.to_excel(writer, sheet_name='All_Methods_Top2_Industry_Combined', index=False)
        
        # Comparison analysis
        if len(all_comparisons) > 0:
            all_comparisons.to_excel(writer, sheet_name='Original_vs_Derived_Analysis', index=False)
        
        # Method parameters summary
        params_summary = pd.DataFrame({
            'Method': list(methods_config.keys()),
            'Parameters': [params for _, params in methods_config.values()],
            'Total_Clusters': [len(np.unique(pred)) for pred, _ in methods_config.values()],
            'Noise_Points': [np.sum(np.array(pred) == -1) for pred, _ in methods_config.values()],
            'Data_Points': [len(pred) for pred, _ in methods_config.values()]
        })
        params_summary.to_excel(writer, sheet_name='Methods_Parameters', index=False)
    
    print("✅ Enhanced Excel created: '../output/complete_top3_aoi_top2_industry_analysis_fixed.xlsx'")
    
except Exception as e:
    print(f"❌ Error creating Excel file: {e}")

# ========================================
# SUMMARY STATISTICS
# ========================================
print(f"\n=== ENHANCED ANALYSIS SUMMARY ===")
print(f"Methods analyzed: {len(methods_config)}")
print(f"Total sheets created: {len([k for k, v in top3_aoi_results.items() if len(v) > 0]) + len([k for k, v in top2_industry_results.items() if len(v) > 0]) + 3}")

if aoi_comparisons:
    print(f"\n=== AOI PATTERN EFFECTIVENESS (vs Original) ===")
    for comparison in aoi_comparisons:
        method = comparison['method']
        effectiveness = comparison['effectiveness_score']
        matching = comparison['matching_patterns']
        similar = comparison['similar_patterns']
        total = comparison['total_original_clusters']
        print(f"{method:12}: {effectiveness:5.1f}% effective ({matching} exact + {similar} similar / {total} total)")

if industry_comparisons:
    print(f"\n=== INDUSTRY PATTERN EFFECTIVENESS (vs Original) ===")
    for comparison in industry_comparisons:
        method = comparison['method']
        effectiveness = comparison['effectiveness_score']
        matching = comparison['matching_patterns']
        similar = comparison['similar_patterns']
        total = comparison['total_original_clusters']
        print(f"{method:12}: {effectiveness:5.1f}% effective ({matching} exact + {similar} similar / {total} total)")

# Find and display best method
if aoi_comparisons and industry_comparisons:
    # Calculate combined effectiveness scores
    method_scores = {}
    for aoi_comp in aoi_comparisons:
        method = aoi_comp['method']
        method_scores[method] = aoi_comp['effectiveness_score']
    
    for ind_comp in industry_comparisons:
        method = ind_comp['method']
        if method in method_scores:
            method_scores[method] = (method_scores[method] + ind_comp['effectiveness_score']) / 2
        else:
            method_scores[method] = ind_comp['effectiveness_score']
    
    if method_scores:
        best_method = max(method_scores.keys(), key=lambda x: method_scores[x])
        best_score = method_scores[best_method]
        
        print(f"\n🏆 BEST METHOD: {best_method}")
        print(f"📊 Combined Effectiveness Score: {best_score:.1f}%")
        
        print(f"\n📈 ALL METHOD SCORES:")
        for method, score in sorted(method_scores.items(), key=lambda x: x[1], reverse=True):
            print(f"  {method:12}: {score:5.1f}%")

print("\n✅ Analysis complete!")


Generating predictions...
✅ Approximate predictions generated
Membership vectors shape: (1569, 22)
Valid clusters count: 17
Clipping indices: membership has 22 columns, but only 17 valid clusters
✅ Membership predictions generated
✅ KNN predictions generated
Generating Top 3 AOI and Top 2 Industry analyses...
  Processing Original...
  ✅ Original analysis completed
  Processing Approximate...
  ✅ Approximate analysis completed
  Processing Membership...
  ✅ Membership analysis completed
  Processing KNN...
  ✅ KNN analysis completed
Creating comparison analyses...
✅ AOI comparisons completed
✅ Industry comparisons completed
Creating Excel file...


c:\Users\abhinash\Documents\Internship\clustering\.venv\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


✅ Enhanced Excel created: '../output/complete_top3_aoi_top2_industry_analysis_fixed.xlsx'

=== ENHANCED ANALYSIS SUMMARY ===
Methods analyzed: 4
Total sheets created: 11

=== AOI PATTERN EFFECTIVENESS (vs Original) ===
Approximate :  72.2% effective (9 exact + 8 similar / 18 total)
Membership  :  69.4% effective (8 exact + 9 similar / 18 total)
KNN         :  88.9% effective (15 exact + 2 similar / 18 total)

=== INDUSTRY PATTERN EFFECTIVENESS (vs Original) ===
Approximate :   0.0% effective (0 exact + 0 similar / 18 total)
Membership  :   0.0% effective (0 exact + 0 similar / 18 total)
KNN         :   0.0% effective (0 exact + 0 similar / 18 total)

🏆 BEST METHOD: KNN
📊 Combined Effectiveness Score: 44.4%

📈 ALL METHOD SCORES:
  KNN         :  44.4%
  Approximate :  36.1%
  Membership  :  34.7%

✅ Analysis complete!


In [213]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
embedding_array=std2dn
# ========================================
# ENHANCED ANALYSIS WITH AUTOMATIC SCORING
# ========================================
def get_top3_aoi_analysis(df, method_predictions, method_name, method_params):
    """Enhanced Top 3 AOI analysis with ranking scores"""
    df_method = df.copy()
    if len(method_predictions) < len(df):
        df_method = df.iloc[:len(method_predictions)].copy()
    
    df_method['cluster'] = method_predictions[:len(df_method)]
    
    # Remove noise points for better analysis
    df_method_clean = df_method[df_method['cluster'] != -1]
    
    # Melt AOI data
    df_melted = df_method_clean.melt(
        id_vars=['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'],
        value_vars=['aoi_1', 'aoi_2', 'aoi_3'],
        var_name='aoi_type',
        value_name='aoi_value'
    )
    
    # Count AOI frequencies with percentage
    aoi_counts = (
        df_melted.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function', 'aoi_value'])
        .size()
        .reset_index(name='count')
    )
    
    # Add percentage within cluster
    cluster_totals = df_melted.groupby('cluster').size().reset_index(name='cluster_total')
    aoi_counts = aoi_counts.merge(cluster_totals, on='cluster')
    aoi_counts['percentage'] = (aoi_counts['count'] / aoi_counts['cluster_total'] * 100).round(2)
    
    # Get Top 3 AOIs per cluster with ranking
    top_3_aoi = (
        aoi_counts.sort_values(['cluster', 'count'], ascending=[True, False])
                  .groupby('cluster')
                  .head(3)
                  .reset_index(drop=True)
    )
    
    # Add ranking within cluster
    top_3_aoi['rank'] = top_3_aoi.groupby('cluster').cumcount() + 1
    top_3_aoi['prediction_method'] = method_name
    top_3_aoi['method_parameters'] = method_params
    
    return top_3_aoi

def get_top2_industry_analysis(df, method_predictions, method_name, method_params):
    """Enhanced Top 2 Industry analysis with ranking scores"""
    df_method = df.copy()
    if len(method_predictions) < len(df):
        df_method = df.iloc[:len(method_predictions)].copy()
    
    df_method['cluster'] = method_predictions[:len(df_method)]
    df_method_clean = df_method[df_method['cluster'] != -1]
    
    # Group by cluster and industry combinations
    industry_counts = (
        df_method_clean.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'])
        .size()
        .reset_index(name='count')
    )
    
    # Add percentage within cluster
    cluster_totals = df_method_clean.groupby('cluster').size().reset_index(name='cluster_total')
    industry_counts = industry_counts.merge(cluster_totals, on='cluster')
    industry_counts['percentage'] = (industry_counts['count'] / industry_counts['cluster_total'] * 100).round(2)
    
    top_2_industries = (
        industry_counts.sort_values(['cluster', 'count'], ascending=[True, False])
        .groupby('cluster')
        .head(2)
        .reset_index(drop=True)
    )
    
    # Add ranking within cluster
    top_2_industries['rank'] = top_2_industries.groupby('cluster').cumcount() + 1
    top_2_industries['prediction_method'] = method_name
    top_2_industries['method_parameters'] = method_params
    
    return top_2_industries

# ========================================
# FIXED PREDICTION METHODS
# ========================================
def get_approximate_predictions(clusterer, embedding_array):
    predicted_labels, strengths = hdbscan.approximate_predict(clusterer, embedding_array)
    params = f"min_samples={clusterer.min_samples}, epsilon={clusterer.cluster_selection_epsilon}"
    return predicted_labels, params

def get_membership_predictions(clusterer, embedding_array, labels):
    """FIXED: Correct shape comparison"""
    membership_vectors = hdbscan.membership_vector(clusterer, embedding_array)
    valid_clusters = np.unique(labels[labels >= 0])
    
    print(f"Membership vectors shape: {membership_vectors.shape}")
    print(f"Valid clusters count: {len(valid_clusters)}")
    
    cluster_indices = np.argmax(membership_vectors, axis=1)
    max_memberships = np.max(membership_vectors, axis=1)
    
    # FIX: Compare number of columns (shape[1]) with number of valid clusters
    if membership_vectors.shape[1] > len(valid_clusters):
        print(f"Clipping indices: membership has {membership_vectors.shape[1]} columns, but only {len(valid_clusters)} valid clusters")
        cluster_indices = np.clip(cluster_indices, 0, len(valid_clusters)-1)
    
    mapped_predictions = valid_clusters[cluster_indices]
    noise_threshold = 0.15
    final_predictions = np.where(max_memberships < noise_threshold, -1, mapped_predictions)
    
    params = f"min_samples={clusterer.min_samples}, noise_threshold=0.15"
    return final_predictions, params

def get_knn_predictions(std2d, labels, embedding_array):
    knn_predictor = KNeighborsClassifier(n_neighbors=5, weights='distance')
    knn_predictor.fit(std2d, labels)
    knn_predictions = knn_predictor.predict(embedding_array)
    
    params = "n_neighbors=5, weights=distance"
    return knn_predictions, params

# ========================================
# SIMPLIFIED COMPARISON METRICS (to avoid complex errors)
# ========================================
def calculate_simple_similarity_scores(original_data, derived_data, analysis_type='aoi'):
    """
    Simplified but robust similarity calculation
    """
    scores = {
        'exact_match_score': 0.0,
        'coverage_score': 0.0,
        'diversity_score': 0.0
    }
    
    try:
        # Get patterns for each method
        if analysis_type == 'aoi':
            orig_patterns = {}
            for _, row in original_data.iterrows():
                cluster = row['cluster']
                if cluster not in orig_patterns:
                    orig_patterns[cluster] = set()
                orig_patterns[cluster].add(row['aoi_value'])
            
            deriv_patterns = {}
            for _, row in derived_data.iterrows():
                cluster = row['cluster']
                if cluster not in deriv_patterns:
                    deriv_patterns[cluster] = set()
                deriv_patterns[cluster].add(row['aoi_value'])
        
        else:  # industry
            orig_patterns = {}
            for _, row in original_data.iterrows():
                cluster = row['cluster']
                if cluster not in orig_patterns:
                    orig_patterns[cluster] = set()
                pattern = f"{row['Parent Industry']}_{row['Job_Seniority']}_{row['Job_Function']}"
                orig_patterns[cluster].add(pattern)
            
            deriv_patterns = {}
            for _, row in derived_data.iterrows():
                cluster = row['cluster']
                if cluster not in deriv_patterns:
                    deriv_patterns[cluster] = set()
                pattern = f"{row['Parent Industry']}_{row['Job_Seniority']}_{row['Job_Function']}"
                deriv_patterns[cluster].add(pattern)
        
        # Calculate similarity scores
        total_clusters = len(orig_patterns)
        exact_matches = 0
        coverage_matches = 0
        
        for orig_cluster, orig_items in orig_patterns.items():
            best_match_score = 0
            
            for deriv_cluster, deriv_items in deriv_patterns.items():
                if len(orig_items.union(deriv_items)) > 0:
                    jaccard = len(orig_items.intersection(deriv_items)) / len(orig_items.union(deriv_items))
                    best_match_score = max(best_match_score, jaccard)
            
            if best_match_score >= 0.8:
                exact_matches += 1
            if best_match_score >= 0.5:
                coverage_matches += 1
        
        if total_clusters > 0:
            scores['exact_match_score'] = (exact_matches / total_clusters) * 100
            scores['coverage_score'] = (coverage_matches / total_clusters) * 100
        
        # Diversity preservation
        all_orig = set()
        all_deriv = set()
        for items in orig_patterns.values():
            all_orig.update(items)
        for items in deriv_patterns.values():
            all_deriv.update(items)
        
        if len(all_orig) > 0:
            scores['diversity_score'] = (len(all_orig.intersection(all_deriv)) / len(all_orig)) * 100
    
    except Exception as e:
        print(f"Error in similarity calculation: {e}")
        # Return default scores on error
    
    return scores

def calculate_clustering_quality_scores(original_labels, derived_labels):
    """
    Simplified clustering quality metrics
    """
    scores = {}
    
    try:
        # Align lengths
        min_len = min(len(original_labels), len(derived_labels))
        orig_aligned = np.array(original_labels)[:min_len]
        deriv_aligned = np.array(derived_labels)[:min_len]
        
        # Only calculate if we have valid data
        if min_len > 0:
            scores['adjusted_rand_score'] = adjusted_rand_score(orig_aligned, deriv_aligned) * 100
            scores['normalized_mutual_info'] = normalized_mutual_info_score(orig_aligned, deriv_aligned) * 100
            
            # Cluster count similarity
            orig_clusters = len(np.unique(orig_aligned))
            deriv_clusters = len(np.unique(deriv_aligned))
            scores['cluster_count_similarity'] = (1 - abs(orig_clusters - deriv_clusters) / max(orig_clusters, deriv_clusters)) * 100
        else:
            scores = {'adjusted_rand_score': 0, 'normalized_mutual_info': 0, 'cluster_count_similarity': 0}
    
    except Exception as e:
        print(f"Error in clustering quality calculation: {e}")
        scores = {'adjusted_rand_score': 0, 'normalized_mutual_info': 0, 'cluster_count_similarity': 0}
    
    return scores

# ========================================
# SIMPLIFIED BEST METHOD SELECTION
# ========================================
def select_best_method_simple(all_scores):
    """
    Simplified method selection with robust error handling
    """
    method_scores = {}
    
    for method_name, scores in all_scores.items():
        if method_name == 'Original':
            continue
        
        # Simple weighted average
        total_score = 0
        count = 0
        
        for score_name, score_value in scores.items():
            if not np.isnan(score_value) and score_value >= 0:
                total_score += score_value
                count += 1
        
        if count > 0:
            method_scores[method_name] = total_score / count
        else:
            method_scores[method_name] = 0
    
    if method_scores:
        best_method = max(method_scores.keys(), key=lambda x: method_scores[x])
        return best_method, method_scores
    else:
        return 'Approximate', {'Approximate': 0}  # Default fallback

# ========================================
# MAIN EXECUTION - SIMPLIFIED & ROBUST
# ========================================

print("Generating predictions...")
original_params = f"min_samples={clusterer.min_samples}, epsilon={clusterer.cluster_selection_epsilon}"

try:
    approx_predictions, approx_params = get_approximate_predictions(clusterer, embedding_array)
    print("✅ Approximate predictions generated")
except Exception as e:
    print(f"❌ Error in approximate predictions: {e}")
    approx_predictions, approx_params = labels, "error"

try:
    membership_predictions, membership_params = get_membership_predictions(clusterer, embedding_array, labels)
    print("✅ Membership predictions generated")
except Exception as e:
    print(f"❌ Error in membership predictions: {e}")
    membership_predictions, membership_params = labels, "error"

try:
    knn_predictions, knn_params = get_knn_predictions(std2d, labels, embedding_array)
    print("✅ KNN predictions generated")
except Exception as e:
    print(f"❌ Error in KNN predictions: {e}")
    knn_predictions, knn_params = labels, "error"

methods_config = {
    'Original': (labels, original_params),
    'Approximate': (approx_predictions, approx_params),
    'Membership': (membership_predictions, membership_params),
    'KNN': (knn_predictions, knn_params)
}

# Generate analyses
print("Generating analyses...")
top3_aoi_results = {}
top2_industry_results = {}

for method_name, (predictions, params) in methods_config.items():
    try:
        print(f"  Processing {method_name}...")
        top3_aoi = get_top3_aoi_analysis(df, predictions, method_name, params)
        top2_industry = get_top2_industry_analysis(df, predictions, method_name, params)
        
        top3_aoi_results[method_name] = top3_aoi
        top2_industry_results[method_name] = top2_industry
        print(f"  ✅ {method_name} analysis complete")
    except Exception as e:
        print(f"  ❌ Error processing {method_name}: {e}")

# Calculate comparison scores
print("Calculating similarity scores...")
all_method_scores = {}

for method_name in methods_config.keys():
    if method_name == 'Original' or method_name not in top3_aoi_results:
        continue
    
    try:
        # AOI similarity scores
        aoi_scores = calculate_simple_similarity_scores(
            top3_aoi_results['Original'], 
            top3_aoi_results[method_name], 
            'aoi'
        )
        
        # Industry similarity scores
        industry_scores = calculate_simple_similarity_scores(
            top2_industry_results['Original'], 
            top2_industry_results[method_name], 
            'industry'
        )
        
        # Clustering quality scores
        clustering_scores = calculate_clustering_quality_scores(
            methods_config['Original'][0], 
            methods_config[method_name]
        )
        
        # Combine all scores
        all_method_scores[method_name] = {
            'aoi_exact_match': aoi_scores['exact_match_score'],
            'aoi_coverage': aoi_scores['coverage_score'],
            'aoi_diversity': aoi_scores['diversity_score'],
            'industry_exact_match': industry_scores['exact_match_score'],
            'industry_coverage': industry_scores['coverage_score'],
            'industry_diversity': industry_scores['diversity_score'],
            'adjusted_rand_score': clustering_scores['adjusted_rand_score'],
            'normalized_mutual_info': clustering_scores['normalized_mutual_info'],
            'cluster_count_similarity': clustering_scores['cluster_count_similarity']
        }
        print(f"  ✅ {method_name} scoring complete")
    
    except Exception as e:
        print(f"  ❌ Error scoring {method_name}: {e}")

# Select best method
print("Selecting best method...")
try:
    best_method, method_rankings = select_best_method_simple(all_method_scores)
    print(f"✅ Best method selected: {best_method}")
except Exception as e:
    print(f"❌ Error in method selection: {e}")
    best_method = 'Approximate'
    method_rankings = {}

# Create Excel output
print("Creating Excel file...")
try:
    with pd.ExcelWriter('../output/robust_method_comparison.xlsx') as writer:
        
        # Method rankings
        if method_rankings:
            ranking_df = pd.DataFrame([
                {'method': method, 'average_score': score}
                for method, score in sorted(method_rankings.items(), key=lambda x: x[1], reverse=True)
            ])
            ranking_df.to_excel(writer, sheet_name='Method_Rankings', index=False)
        
        # Individual method sheets
        for method_name, top3_data in top3_aoi_results.items():
            if len(top3_data) > 0:
                sheet_name = f'Top3_AOI_{method_name}'[:31]  # Excel sheet name limit
                top3_data.to_excel(writer, sheet_name=sheet_name, index=False)
        
        for method_name, top2_data in top2_industry_results.items():
            if len(top2_data) > 0:
                sheet_name = f'Top2_Ind_{method_name}'[:31]  # Excel sheet name limit
                top2_data.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Best method highlight
        if best_method in top3_aoi_results:
            best_aoi = top3_aoi_results[best_method]
            best_aoi.to_excel(writer, sheet_name=f'BEST_AOI_{best_method}'[:31], index=False)
        
        if best_method in top2_industry_results:
            best_industry = top2_industry_results[best_method]
            best_industry.to_excel(writer, sheet_name=f'BEST_IND_{best_method}'[:31], index=False)
    
    print("✅ Excel file created successfully!")
    
except Exception as e:
    print(f"❌ Error creating Excel file: {e}")

print(f"\n🏆 BEST METHOD: {best_method}")
if method_rankings:
    print(f"📊 METHOD SCORES:")
    for method, score in sorted(method_rankings.items(), key=lambda x: x[1], reverse=True):
        print(f"  {method:12}: {score:6.2f}")


Generating predictions...
✅ Approximate predictions generated
Membership vectors shape: (1569, 22)
Valid clusters count: 17
Clipping indices: membership has 22 columns, but only 17 valid clusters
✅ Membership predictions generated
✅ KNN predictions generated
Generating analyses...
  Processing Original...
  ✅ Original analysis complete
  Processing Approximate...
  ✅ Approximate analysis complete
  Processing Membership...
  ✅ Membership analysis complete
  Processing KNN...
  ✅ KNN analysis complete
Calculating similarity scores...
Error in clustering quality calculation: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.
  ✅ Approximate scoring complete
Error in clustering quality calculation: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.
  ✅ Membership scoring compl

In [217]:
import pandas as pd
import numpy as np
import hdbscan

# ========================================
# DIAGNOSE THE CLUSTER MISMATCH
# ========================================
print("=== CLUSTER ALIGNMENT DIAGNOSIS ===")
print(f"Original clusters: {sorted(np.unique(labels))}")
print(f"Predicted clusters: {sorted(np.unique(predicted_labels))}")

# Check which clusters are missing in predictions
original_clusters = set(np.unique(labels))
predicted_clusters = set(np.unique(predicted_labels))

missing_in_predictions = original_clusters - predicted_clusters
extra_in_predictions = predicted_clusters - original_clusters

print(f"Missing in predictions: {missing_in_predictions}")
print(f"Extra in predictions: {extra_in_predictions}")

# ========================================
# FIX 1: ALIGN CLUSTER NUMBERS
# ========================================
def align_predicted_clusters_with_original(original_labels, predicted_labels, embedding_array, clusterer):
    """
    Ensure predicted cluster numbers align with original cluster numbers
    """
    from sklearn.neighbors import NearestNeighbors
    
    # Get original cluster centroids
    original_clusters = np.unique(original_labels[original_labels != -1])
    cluster_centroids = {}
    
    for cluster_id in original_clusters:
        cluster_mask = original_labels == cluster_id
        cluster_points = std2d[cluster_mask]  # Use your training data
        centroid = np.mean(cluster_points, axis=0)
        cluster_centroids[cluster_id] = centroid
    
    # For each predicted point, find closest original cluster centroid
    aligned_predictions = []
    aligned_strengths = []
    
    for i, (pred_cluster, point) in enumerate(zip(predicted_labels, embedding_array)):
        if pred_cluster == -1:  # Keep noise as noise
            aligned_predictions.append(-1)
            aligned_strengths.append(strengths[i])
        else:
            # Find closest original cluster centroid
            min_distance = float('inf')
            closest_cluster = -1
            
            for orig_cluster_id, centroid in cluster_centroids.items():
                distance = np.linalg.norm(point - centroid)
                if distance < min_distance:
                    min_distance = distance
                    closest_cluster = orig_cluster_id
            
            # Assign to closest original cluster
            aligned_predictions.append(closest_cluster)
            aligned_strengths.append(strengths[i])
    
    return np.array(aligned_predictions), np.array(aligned_strengths)

# Apply alignment
print("Aligning predicted clusters with original clusters...")
aligned_predictions, aligned_strengths = align_predicted_clusters_with_original(
    labels, predicted_labels, embedding_array, clusterer
)

print(f"Aligned predictions clusters: {sorted(np.unique(aligned_predictions))}")

# ========================================
# FIX 2: ENHANCED ANALYSIS FUNCTIONS WITH ALIGNMENT
# ========================================
def get_aligned_top3_aoi_analysis(df, method_predictions, method_name, params, reference_clusters=None):
    """
    Get Top 3 AOI analysis ensuring all reference clusters are represented
    """
    df_method = df.copy()
    if len(method_predictions) < len(df):
        df_method = df.iloc[:len(method_predictions)].copy()
    
    df_method['cluster'] = method_predictions[:len(df_method)]
    
    # Melt AOI data
    df_melted = df_method.melt(
        id_vars=['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'],
        value_vars=['aoi_1', 'aoi_2', 'aoi_3'],
        var_name='aoi_type',
        value_name='aoi_value'
    )
    
    # Count AOI frequencies
    aoi_counts = (
        df_melted.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function', 'aoi_value'])
        .size()
        .reset_index(name='count')
    )
    
    # Get Top 3 AOIs per cluster
    top_3_aoi = (
        aoi_counts.sort_values(['cluster', 'count'], ascending=[True, False])
                  .groupby('cluster')
                  .head(3)
                  .reset_index(drop=True)
    )
    
    # Add method information
    top_3_aoi['prediction_method'] = method_name
    top_3_aoi['method_parameters'] = params
    
    # Add cluster coverage info
    predicted_cluster_set = set(method_predictions[method_predictions != -1])
    if reference_clusters is not None:
        reference_cluster_set = set(reference_clusters[reference_clusters != -1])
        coverage = len(predicted_cluster_set.intersection(reference_cluster_set)) / len(reference_cluster_set)
        top_3_aoi['cluster_coverage'] = f"{coverage:.2%}"
    
    return top_3_aoi

def get_aligned_top2_industry_analysis(df, method_predictions, method_name, params, reference_clusters=None):
    """
    Get Top 2 Industry combinations per cluster with alignment info
    """
    df_method = df.copy()
    if len(method_predictions) < len(df):
        df_method = df.iloc[:len(method_predictions)].copy()
    
    df_method['cluster'] = method_predictions[:len(df_method)]
    
    # Group by cluster and industry combinations
    top_2_industries = (
        df_method.groupby(['cluster', 'Parent Industry', 'Job_Seniority', 'Job_Function'])
        .size()
        .reset_index(name='count')
        .sort_values(['cluster', 'count'], ascending=[True, False])
        .groupby('cluster')
        .head(2)
        .reset_index(drop=True)
    )
    
    # Add method information
    top_2_industries['prediction_method'] = method_name
    top_2_industries['method_parameters'] = params
    
    # Add cluster coverage info
    predicted_cluster_set = set(method_predictions[method_predictions != -1])
    if reference_clusters is not None:
        reference_cluster_set = set(reference_clusters[reference_clusters != -1])
        coverage = len(predicted_cluster_set.intersection(reference_cluster_set)) / len(reference_cluster_set)
        top_2_industries['cluster_coverage'] = f"{coverage:.2%}"
    
    return top_2_industries

# ========================================
# RUN ALIGNED ANALYSIS
# ========================================

method_name = 'approximate_predict_aligned'
params = f"min_samples={clusterer.min_samples}, epsilon={clusterer.cluster_selection_epsilon}, aligned=True"

# Original clustering analysis  
print("Analyzing original clustering...")
top3_aoi_original = get_aligned_top3_aoi_analysis(df, labels, 'original', params)
top2_industry_original = get_aligned_top2_industry_analysis(df, labels, 'original', params)

# Aligned predictions analysis
print("Analyzing aligned predictions...")
top3_aoi_aligned = get_aligned_top3_aoi_analysis(df, aligned_predictions, method_name, params, labels)
top2_industry_aligned = get_aligned_top2_industry_analysis(df, aligned_predictions, method_name, params, labels)

# ========================================
# CLUSTER MATCHING COMPARISON
# ========================================
def compare_cluster_patterns(original_aoi, predicted_aoi, analysis_type='AOI'):
    """
    Compare cluster patterns between original and predicted
    """
    comparison_results = []
    
    # Get clusters present in both
    orig_clusters = set(original_aoi['cluster'].unique())
    pred_clusters = set(predicted_aoi['cluster'].unique())
    common_clusters = orig_clusters.intersection(pred_clusters)
    
    for cluster_id in sorted(common_clusters):
        if cluster_id == -1:  # Skip noise
            continue
            
        # Get patterns for this cluster
        orig_patterns = set(original_aoi[original_aoi['cluster'] == cluster_id]['aoi_value' if analysis_type == 'AOI' else 'Parent Industry'])
        pred_patterns = set(predicted_aoi[predicted_aoi['cluster'] == cluster_id]['aoi_value' if analysis_type == 'AOI' else 'Parent Industry'])
        
        # Calculate similarity
        if len(orig_patterns.union(pred_patterns)) > 0:
            similarity = len(orig_patterns.intersection(pred_patterns)) / len(orig_patterns.union(pred_patterns))
        else:
            similarity = 0
        
        comparison_results.append({
            'cluster': cluster_id,
            'original_patterns': len(orig_patterns),
            'predicted_patterns': len(pred_patterns),
            'common_patterns': len(orig_patterns.intersection(pred_patterns)),
            'similarity_score': similarity,
            'match_quality': 'High' if similarity >= 0.8 else 'Medium' if similarity >= 0.5 else 'Low'
        })
    
    return pd.DataFrame(comparison_results)

# Compare AOI patterns
aoi_comparison = compare_cluster_patterns(top3_aoi_original, top3_aoi_aligned, 'AOI')

# Compare Industry patterns  
industry_comparison = compare_cluster_patterns(top2_industry_original, top2_industry_aligned, 'Industry')

# ========================================
# SAVE ALIGNED RESULTS
# ========================================
print("Creating aligned Excel file...")
with pd.ExcelWriter('../output/aligned_cluster_analysis.xlsx') as writer:
    
    # Original clustering results
    top3_aoi_original.to_excel(writer, sheet_name='Top3_AOI_Original', index=False)
    top2_industry_original.to_excel(writer, sheet_name='Top2_Industry_Original', index=False)
    
    # Aligned prediction results
    top3_aoi_aligned.to_excel(writer, sheet_name='Top3_AOI_Aligned', index=False)
    top2_industry_aligned.to_excel(writer, sheet_name='Top2_Industry_Aligned', index=False)
    
    # Cluster pattern comparisons
    aoi_comparison.to_excel(writer, sheet_name='AOI_Pattern_Comparison', index=False)
    industry_comparison.to_excel(writer, sheet_name='Industry_Pattern_Comparison', index=False)
    
    # Alignment details
    alignment_details = pd.DataFrame({
        'point_id': range(len(aligned_predictions)),
        'original_prediction': predicted_labels,
        'aligned_prediction': aligned_predictions,
        'prediction_strength': aligned_strengths,
        'alignment_changed': predicted_labels != aligned_predictions
    })
    alignment_details.to_excel(writer, sheet_name='Alignment_Details', index=False)
    
    # Summary
    summary = pd.DataFrame({
        'Metric': [
            'Original Clusters',
            'Predicted Clusters (before alignment)',
            'Predicted Clusters (after alignment)', 
            'Cluster Alignment Changes',
            'AOI Pattern Matches (High)',
            'Industry Pattern Matches (High)'
        ],
        'Value': [
            len(np.unique(labels[labels != -1])),
            len(np.unique(predicted_labels[predicted_labels != -1])),
            len(np.unique(aligned_predictions[aligned_predictions != -1])),
            np.sum(predicted_labels != aligned_predictions),
            np.sum(aoi_comparison['match_quality'] == 'High'),
            np.sum(industry_comparison['match_quality'] == 'High')
        ]
    })
    summary.to_excel(writer, sheet_name='Alignment_Summary', index=False)

print("✅ Aligned cluster analysis saved: '../output/aligned_cluster_analysis.xlsx'")

# ========================================
# ALIGNMENT SUMMARY
# ========================================
print(f"\n=== CLUSTER ALIGNMENT RESULTS ===")
print(f"Original clusters: {sorted(np.unique(labels[labels != -1]))}")
print(f"Predicted (before): {sorted(np.unique(predicted_labels[predicted_labels != -1]))}")
print(f"Predicted (after): {sorted(np.unique(aligned_predictions[aligned_predictions != -1]))}")

print(f"\nAlignment changes: {np.sum(predicted_labels != aligned_predictions)} points")
print(f"Perfect alignment: {'✅' if set(np.unique(aligned_predictions)) == set(np.unique(labels)) else '❌'}")

print(f"\nPattern matching quality:")
print(f"AOI patterns - High matches: {np.sum(aoi_comparison['match_quality'] == 'High')}/{len(aoi_comparison)}")
print(f"Industry patterns - High matches: {np.sum(industry_comparison['match_quality'] == 'High')}/{len(industry_comparison)}")


=== CLUSTER ALIGNMENT DIAGNOSIS ===
Original clusters: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]
Predicted clusters: [np.int32(-1), np.int32(10), np.int32(20), np.int32(28), np.int32(29), np.int32(35), np.int32(56), np.int32(81)]
Missing in predictions: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)}
Extra in predictions: {np.int32(35), np.int32(81), np.int32(20), np.int32(56), np.int32(28), np.int32(29)}
Aligning predicted clusters with original clusters...
Aligned predictions clusters: [np.int64(-1), np.int64(4), np.int64(5), np.int64(8), np.int64(14)]
Analyzing original clustering...
Analyzing aligned predictions...
Creat